# Coupled 0-π qubits

Two symmetric 0-π qubits couple through capacitors $C_c$ to two shared coupler nodes, following [Liu, Weissler, and Combes, arXiv:2609.16630 (2026)](https://doi.org/10.48550/arXiv.2609.16630).
Each qubit node also has a capacitance $C_0$ to ground.
This is scheme (a) of that paper, a pure $n_{\theta_1} n_{\theta_2}$ coupler.
The example quantizes the circuit in the paper's coordinates and reproduces its coupling energy $g_\theta$.

In [ ]:
import sympy as sym
import circuit_quantization as cq
from circuit_quantization import Branch, Circuit, Loops, quantize
from circuit_quantization.units import e, h_GHz as h

## Circuit

Qubit 1 has nodes 1–4 and qubit 2 nodes 5–8; nodes 9 and 10 are the coupler and node 0 is ground.
Nodes 1, 4, 5, 8 couple to node 10, and nodes 2, 3, 6, 7 to node 9.

In [ ]:
C_J, C, L, C_0, C_c, E_J = sym.symbols("C_J C L C_0 C_c E_J", positive=True)

branches = []
for q, (a, b, c, d) in enumerate(((1, 2, 3, 4), (5, 6, 7, 8)), start=1):
    branches += [
        Branch(f"J{q}a", a, b, C=C_J, EJ=E_J), Branch(f"J{q}b", c, d, C=C_J, EJ=E_J),
        Branch(f"L{q}a", b, c, L=L), Branch(f"L{q}b", d, a, L=L),
        Branch(f"C{q}a", a, c, C=C), Branch(f"C{q}b", b, d, C=C),
        *(Branch(f"C0_{n}", n, 0, C=C_0) for n in (a, b, c, d)),
    ]
branches += [Branch(f"Cc_{n}", n, 10, C=C_c) for n in (1, 4, 5, 8)]
branches += [Branch(f"Cc_{n}", n, 9, C=C_c) for n in (2, 3, 6, 7)]
coupled = Circuit(branches, ground=0)

Each grounded node gets its own ground symbol, placed outward from the center of the drawing:

In [ ]:
layout = {1: (-2, 1), 2: (-1, 1), 3: (-1, 0), 4: (-2, 0), 5: (1, 1), 6: (2, 1), 7: (2, 0), 8: (1, 0),
          9: (0, 1.6), 10: (0, -0.6)}
cq.draw_circuit_diagram(coupled, layout=layout, grounds="radial", label=False)

## Loops

`from_circuit` builds a loop basis from a spanning tree.
With `zero_capacitive=True` only the two superconducting rings keep a symbolic flux, named after the branch that closes each one.

In [ ]:
loops = Loops.from_circuit(coupled, zero_capacitive=True)

## The paper's coordinates

Each column of `Z` writes one coordinate in terms of the node phases 1–10.
For each qubit the columns are $\theta$, $\Sigma$, $\phi$, and $\zeta$; the last two columns are the coupler modes $\varphi_+'$ and $\varphi_-'$.
The coupler rows include the shift $\varphi_-' = \varphi_- - (\theta_1 + \theta_2)/2$ of the paper's Appendix C, which removes the charge coupling between $\varphi_-$ and the $\theta$ modes.
$\Sigma_1$, $\Sigma_2$, $\varphi_+'$, and $\varphi_-'$ have no potential energy, so they are free modes (`fr`) and `quantize` removes them.

In [ ]:
half, quarter = sym.Rational(1, 2), sym.Rational(1, 4)
qubit = sym.Matrix([[-1, 1, -1, 1], [1, 1, 1, 1], [1, 1, -1, -1], [-1, 1, 1, -1]]) * half  # theta, Sigma, phi, zeta
Z = sym.zeros(10, 10)
Z[0:4, 0:4] = qubit
Z[4:8, 4:8] = qubit
Z[0:8, 8] = sym.ones(8, 1) * half
Z[8, :] = sym.Matrix([[quarter, 0, 0, 0, quarter, 0, 0, 0, half, half]])
Z[9, :] = sym.Matrix([[-quarter, 0, 0, 0, -quarter, 0, 0, 0, half, -half]])
types = ("c", "fr", "e", "h", "c", "fr", "e", "h", "fr", "fr")
Z

## Quantization

In [ ]:
result = quantize(coupled, loops, Z, types, simplify=True)
assert result.coords.types == ("c", "e", "h", "c", "e", "h")  # theta1, phi1, zeta1, theta2, phi2, zeta2
result

Each qubit keeps its $\theta$ (`c`), $\phi$ (`e`), and $\zeta$ (`h`).
The charging-energy matrix couples the two qubits only through $\theta_1$ and $\theta_2$:

In [ ]:
E_C = result.ham.E_C
E_C.applyfunc(sym.factor)

## The coupling energy

The paper writes the mode capacitances as (its Eq. C7)
$C_\theta = 2C + 2C_J + C_c + C_0$, $C_\phi = 2C_J + C_c + C_0$, and $C_\zeta = 2C + C_c + C_0$,
and finds

$$
H_\mathrm{int} = g_\theta n_{\theta_1} n_{\theta_2}, \qquad
g_\theta = \frac{2e^2 C_c}{C_\theta (C_\theta - C_c)},
$$

$$
\tilde E_C^\theta = \frac{e^2 (2C_\theta - C_c)}{4 C_\theta (C_\theta - C_c)},
$$

$$
\tilde E_C^\phi = \frac{e^2}{2C_\phi}, \qquad
\tilde E_C^\zeta = \frac{e^2}{2C_\zeta}.
$$

With $H/h = 4\,\mathbf n^\mathsf{T} E_C \mathbf n + \dots$, the coupling is $g_\theta/h = 8 E_C[\theta_1, \theta_2]$ and each charging energy is a diagonal entry of $E_C$.
All of them agree exactly:

In [ ]:
C_theta = 2*C + 2*C_J + C_c + C_0
C_phi = 2*C_J + C_c + C_0
C_zeta = 2*C + C_c + C_0

g_theta = 2 * e**2 * C_c / (C_theta * (C_theta - C_c))
E_C_theta = e**2 * (2*C_theta - C_c) / (4 * C_theta * (C_theta - C_c))

theta1, phi1, zeta1, theta2 = 0, 1, 2, 3
assert sym.simplify(8 * E_C[theta1, theta2] - g_theta / h) == 0
assert sym.simplify(E_C[theta1, theta1] - E_C_theta / h) == 0
assert sym.simplify(E_C[phi1, phi1] - e**2 / (2 * C_phi) / h) == 0
assert sym.simplify(E_C[zeta1, zeta1] - e**2 / (2 * C_zeta) / h) == 0
sym.Eq(sym.Symbol(r"g_\theta") / h, sym.factor(8 * E_C[theta1, theta2]))

For $C \gg C_c, C_J, C_0$ this reduces to the paper's $g_\theta \approx e^2 C_c / 2C^2$:

In [ ]:
x = sym.Symbol("x", positive=True)  # x = 1/C
leading = sym.limit((g_theta * C**2).subs(C, 1 / x), x, 0)
assert sym.simplify(leading - e**2 * C_c / 2) == 0
leading / C**2